# Understanding _rescued_data Column

## Overview
The `_rescued_data` column automatically captures data that doesn't conform to the defined schema. This happens when:
- Extra columns are present in source data
- Column count mismatches occur
- Data type conversions fail
- Quoted fields contain unexpected delimiters

## Why _rescued_data Matters
- **Data Loss Prevention**: No data is discarded
- **Debugging**: Easy to identify schema mismatches
- **Data Quality**: Detect quality issues automatically
- **Schema Evolution**: Handle changes gradually
- **Compliance**: Maintains audit trail

## Step 1: Import Libraries

In [ ]:
from pyspark.sql.functions import current_timestamp, col, when
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, DateType

## Step 2: Define Expected Schema

In [ ]:
schema = StructType([
    StructField("transaction_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("transaction_date", DateType(), True),
    StructField("payment_method", StringType(), True),
    StructField("status", StringType(), True)
])
print("Expected schema defined")

## Step 3: Create Table to Capture Rescued Data

In [ ]:
DROP TABLE IF EXISTS transactions_with_rescued_data;

CREATE TABLE transactions_with_rescued_data (
  transaction_id STRING,
  customer_id STRING,
  amount DOUBLE,
  transaction_date DATE,
  payment_method STRING,
  status STRING,
  ingestion_timestamp TIMESTAMP,
  _rescued_data STRING
)
USING DELTA

## Step 4: COPY INTO with RESCUEDATACOLUMN

Load CSV with extra/mismatched columns to demonstrate _rescued_data.

In [ ]:
COPY INTO transactions_with_rescued_data
FROM (
  SELECT 
    transaction_id,
    customer_id,
    amount,
    CAST(transaction_date AS DATE) as transaction_date,
    payment_method,
    status,
    CURRENT_TIMESTAMP() as ingestion_timestamp,
    _rescued_data
  FROM read_files(
    '/Volumes/data_analysis/sample_data/transactions_malformed.csv',
    format => 'csv'
  )
)
FILEFORMAT = CSV
COPY_OPTIONS ('rescuedatacolumn' = '_rescued_data')

## Step 5: View All Data

In [ ]:
SELECT * FROM transactions_with_rescued_data

## Step 6: View Records with Rescued Data

In [ ]:
SELECT 
  transaction_id,
  customer_id,
  _rescued_data
FROM transactions_with_rescued_data
WHERE _rescued_data IS NOT NULL

## Step 7: Data Quality Analysis

In [ ]:
SELECT 
  COUNT(*) as total_records,
  SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) as rescued_records,
  SUM(CASE WHEN _rescued_data IS NULL THEN 1 ELSE 0 END) as clean_records,
  ROUND(SUM(CASE WHEN _rescued_data IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) as error_rate_percent
FROM transactions_with_rescued_data

## Step 8: Auto Loader with _rescued_data

In [ ]:
df_stream = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation", "/tmp/autoloader_checkpoint/rescue/") \
    .option("rescuedatacolumn", "_rescued_data") \
    .schema(schema) \
    .option("header", "true") \
    .load("/Volumes/data_analysis/sample_data/")

df_with_meta = df_stream.withColumn("ingestion_timestamp", current_timestamp())
print("Auto Loader stream configured with _rescued_data support")

## Step 9: Write Stream to Delta

In [ ]:
streaming_query = df_with_meta.writeStream \
    .format("delta") \
    .mode("append") \
    .option("checkpointLocation", "/tmp/autoloader_checkpoint/rescue/") \
    .table("transactions_stream_rescue")

print(f"Streaming query started: {streaming_query.id}")

## Step 10: Create Quarantine Table

In [ ]:
CREATE TABLE IF NOT EXISTS transactions_quarantine AS
SELECT 
  transaction_id,
  customer_id,
  amount,
  transaction_date,
  payment_method,
  status,
  _rescued_data,
  ingestion_timestamp,
  'SCHEMA_MISMATCH' as quarantine_reason
FROM transactions_with_rescued_data
WHERE _rescued_data IS NOT NULL

## Step 11: Process Records by Quality

In [ ]:
df = spark.table("transactions_with_rescued_data")

clean_records = df.filter(col("_rescued_data").isNull())
rescued_records = df.filter(col("_rescued_data").isNotNull())

print(f"Clean records: {clean_records.count()}")
print(f"Records requiring investigation: {rescued_records.count()}")

## Step 12: Data Quality Monitoring

In [ ]:
df = spark.table("transactions_with_rescued_data")
total = df.count()
errors = df.filter(col("_rescued_data").isNotNull()).count()
error_rate = (errors / total * 100) if total > 0 else 0

print(f"Error Rate: {error_rate:.2f}%")

if error_rate > 5:
    print("WARNING: Error rate exceeds 5% threshold!")
else:
    print("OK: Error rate is acceptable")

## Common Scenarios Causing _rescued_data

### 1. Extra Columns in Source Data

In [ ]:
-- CSV has (A, B, C, D, E) but schema only defines (A, B, C)
-- Columns D and E are captured in _rescued_data
SELECT 
  transaction_id,
  customer_id,
  _rescued_data
FROM transactions_with_rescued_data
WHERE _rescued_data IS NOT NULL
LIMIT 5

### 2. Inconsistent Column Count Per Row

In [ ]:
-- Different rows have different numbers of columns
-- Extra columns get captured
SELECT 
  transaction_id,
  _rescued_data,
  LENGTH(_rescued_data) as rescued_data_length
FROM transactions_with_rescued_data

## Best Practices

1. Always enable _rescued_data in production pipelines
2. Monitor error rates with dashboards
3. Investigate problematic records regularly
4. Implement quarantine tables
5. Set up alerts for high error rates
6. Update schema to accommodate valid new fields

## Summary

The `_rescued_data` column provides:
- Complete data capture without loss
- Automatic quality monitoring
- Transparent schema mismatch handling
- Foundation for robust error handling

Always enable _rescued_data in production for data integrity and quality management.